# Email Spam Detection
## Exploratory Data Analysis & Model Training
---
**Dataset:** SMS Spam Collection (Kaggle)  
**Models:** Naive Bayes | SVM | Neural Network  
**Goal:** Classify messages as Spam or Ham using NLP + ML  

## 1. Import Libraries

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))  # allow src imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re, nltk
nltk.download('stopwords', quiet=True)

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, confusion_matrix, classification_report)

sns.set_theme(style='darkgrid')
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded.')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('../data/spam.csv', encoding='latin-1')[['v1', 'v2']]
df.columns = ['label', 'text']
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

print(f'Total samples : {len(df)}')
print(f'Spam messages : {df.label_num.sum()}')
print(f'Ham  messages : {(df.label_num == 0).sum()}')
df.head()

## 3. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
counts = df['label'].value_counts()
axes[0].bar(counts.index, counts.values, color=['#43a047', '#e53935'], width=0.4)
axes[0].set_title('Message Count by Class')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=counts.index,
            colors=['#43a047', '#e53935'], autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Class Ratio')

plt.tight_layout()
plt.savefig('../reports/eda_class_distribution.png')
plt.show()

## 4. Message Length Analysis

In [ ]:
df['msg_length'] = df['text'].apply(len)
df['word_count'] = df['text'].apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for label, color in [('ham', '#43a047'), ('spam', '#e53935')]:
    subset = df[df['label'] == label]
    axes[0].hist(subset['msg_length'], bins=40, alpha=0.6, label=label, color=color)
    axes[1].hist(subset['word_count'], bins=30, alpha=0.6, label=label, color=color)

axes[0].set_title('Character Length Distribution')
axes[0].set_xlabel('Characters')
axes[0].legend()

axes[1].set_title('Word Count Distribution')
axes[1].set_xlabel('Words')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/eda_length_distribution.png')
plt.show()

print(df.groupby('label')[['msg_length', 'word_count']].mean().round(1))

## 5. Top Words per Class

In [ ]:
from nltk.corpus import stopwords
STOP = set(stopwords.words('english'))

def top_words(series, n=15):
    words = ' '.join(series).lower().split()
    words = [w for w in words if w.isalpha() and w not in STOP]
    return Counter(words).most_common(n)

ham_top  = top_words(df[df['label'] == 'ham']['text'])
spam_top = top_words(df[df['label'] == 'spam']['text'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, data, color, title in [
    (axes[0], ham_top,  '#43a047', 'Top 15 Ham Words'),
    (axes[1], spam_top, '#e53935', 'Top 15 Spam Words'),
]:
    words, counts = zip(*data)
    ax.barh(words[::-1], counts[::-1], color=color)
    ax.set_title(title)
    ax.set_xlabel('Frequency')

plt.tight_layout()
plt.savefig('../reports/eda_top_words.png')
plt.show()

## 6. Preprocessing & TF-IDF Vectorization

In [ ]:
from src.preprocess import load_and_preprocess

X_train, X_test, y_train, y_test, vectorizer = load_and_preprocess('../data/spam.csv')

print(f'Training samples : {X_train.shape[0]}')
print(f'Test samples     : {X_test.shape[0]}')
print(f'Features (TF-IDF): {X_train.shape[1]}')

## 7. Train & Evaluate All Models

In [ ]:
from src.train    import train_all
from src.evaluate import plot_model_comparison

results = train_all('../data/spam.csv')
print('\nFinal Results:')
for name, m in results.items():
    print(f'  {name:<18} Accuracy={m["accuracy"]:.4f}  F1={m["f1"]:.4f}  Precision={m["precision"]:.4f}  Recall={m["recall"]:.4f}')

## 8. Model Comparison Chart

In [ ]:
from IPython.display import Image
Image('../reports/model_comparison.png')

## 9. Live Prediction Test

In [ ]:
from src.predict import predict

test_messages = [
    'Congratulations! You have won a FREE iPhone. Click now!',
    'Hey, are we still meeting tomorrow at 3pm?',
    'URGENT: Your bank account will be suspended. Verify immediately.',
    'The report is attached, please review by Friday.',
    'Win cash prizes! Call now to claim your reward!',
]

print(f"{'Message':<55} {'Model':<10} {'Label':<6} {'Confidence'}" )
print('-' * 90)
for msg in test_messages:
    for model in ['naive_bayes', 'svm', 'neural_network']:
        r = predict(msg, model_name=model)
        short_msg = msg[:52] + '...' if len(msg) > 52 else msg
        print(f"{short_msg:<55} {model:<18} {r['label']:<6} {r['confidence']}%")
    print()

## 10. Conclusion

| Model | Strength | Best Use |
|-------|----------|----------|
| Naive Bayes | Fast, lightweight | Baseline / low-resource |
| SVM | High precision | Production spam filter |
| Neural Network | Balanced recall | Complex/mixed datasets |

**SVM and Neural Network** both achieve ~98% accuracy on this dataset.  
**SVM** is preferred for production due to consistent precision with real confidence scores.